# 05 — Inference on New Data & GIS Export
Vellakkal Dumpsite methane hotspot project.

Runs the trained model on new points (or re-exports the held-out test predictions), then produces QGIS-ready outputs: a point GeoJSON and an IDW-interpolated probability surface GeoTIFF.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/methane-hotspot-detection'
import os
os.makedirs(f'{PROJECT_ROOT}/outputs/gis', exist_ok=True)

In [ ]:
!pip install -q earthengine-api geopandas rasterio scipy

In [ ]:
import ee
GEE_PROJECT = 'YOUR-GEE-PROJECT-ID'  # <-- same project as earlier batches
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)

import sys
sys.path.append(f'{PROJECT_ROOT}/src')

## Option A — Score new points
Provide a CSV of new (latitude, longitude, date) points to score — e.g. a fresh grid around the dumpsite for a current date, or a monitoring run for a later month.

In [ ]:
NEW_POINTS_CSV = f'{PROJECT_ROOT}/data/raw/new_points_to_score.csv'  # <-- prepare this file
import glob
MODEL_DIR = sorted(glob.glob(f'{PROJECT_ROOT}/models/cnn_v1_*'))[-1]
print('Using model:', MODEL_DIR)

In [ ]:
!python {PROJECT_ROOT}/src/inference.py \
    --points-csv {NEW_POINTS_CSV} \
    --model-dir {MODEL_DIR} \
    --gee-project {GEE_PROJECT} \
    --out-dir {PROJECT_ROOT}/outputs/predictions

## Option B — Use the existing held-out test predictions instead
If you just want to visualize the Batch 6 evaluation results on a map rather than scoring brand-new points, join the test predictions with their coordinates.

In [ ]:
PREDICTIONS_FOR_GIS = f'{PROJECT_ROOT}/outputs/predictions/test_predictions.csv'
METADATA_CSV = f'{PROJECT_ROOT}/data/processed/metadata_with_splits.csv'
# Use this pair for Option B, or switch PREDICTIONS_FOR_GIS to
# inference_predictions.csv (no --metadata-csv needed) for Option A.

## Export GIS layers

In [ ]:
!python {PROJECT_ROOT}/src/gis_export.py \
    --predictions-csv {PREDICTIONS_FOR_GIS} \
    --metadata-csv {METADATA_CSV} \
    --out-dir {PROJECT_ROOT}/outputs/gis \
    --grid-res-deg 0.002

## Quick in-notebook preview
A fast look before opening the layers in QGIS.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show

gdf = gpd.read_file(f'{PROJECT_ROOT}/outputs/gis/predictions_points.geojson')
fig, ax = plt.subplots(1, 2, figsize=(13, 6))

gdf.plot(column='hotspot_probability', cmap='coolwarm', legend=True, markersize=15, ax=ax[0])
ax[0].set_title('Point predictions')

with rasterio.open(f'{PROJECT_ROOT}/outputs/gis/predictions_probability_surface.tif') as src:
    show(src, ax=ax[1], cmap='coolwarm')
ax[1].set_title('IDW-interpolated surface (visualization aid, not raw data)')
plt.tight_layout()
plt.show()

## Open in QGIS
Download `predictions_points.geojson` and `predictions_probability_surface.tif` from Drive and load them as layers in QGIS, overlaid on a Sentinel-2 true-color basemap or OpenStreetMap, for the final GIS deliverable.